In [1]:
from pandas import read_csv, concat, DataFrame, to_datetime
from matplotlib import pyplot as plt
import seaborn as sns; sns.set_theme()
from os.path import expanduser
from os import stat
from glob import glob
from pandas import Timestamp
from numpy import sin, cos, pi
from numpy import array, copy, mean, std
from tensorflow.keras.models import load_model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, InputLayer
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.regularizers import l2
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.metrics import RootMeanSquaredError
from tensorflow.keras.optimizers import Adam


In [58]:
def get_files(fn):
    path = r'Datasets/DEVRT/'
    files = glob(path + fn + '/*.csv')
    return files


def to_date(char):
    return char[:4] + '-' + char[4:6] + '-' + char[6:8]


# for each file create a pandas df
# concatenate them all together
# change index to datetime
def parse_file(files, filenames):
    list_of_pandas = []
    for i, f in enumerate(files):
        # if file empty
        if stat(f).st_size != 0:
            # create df
            temp = read_csv(f, header=0, nrows=288, skip_blank_lines=False)
            # create date column
            temp['Date'] = [to_date(filenames[i]) for _ in range(288)]
            # append df 
            list_of_pandas += [temp]
    # cast all together
    df = concat(list_of_pandas, ignore_index=True)
# Change line 27 in parse_file to:
    df.index = to_datetime(df['Date'] + ' ' + df['Time'], format='%Y-%m-%d %H:%M')
    df.drop(columns=['Date', 'Time'], inplace=True)
    return df

In [59]:
# list files
# demand_files = get_files('demand')
source_files = get_files('NISSAN_LEAF_TRAIN')

# list filenames to use on timestamps
filenames = [file[-12: -4] for file in source_files]

# get data from files
# demands = parse_file(demand_files, filenames)
sources = parse_file(source_files, filenames)

ValueError: Length of values (288) does not match length of index (279)

In [14]:
# move values and drop temp cols
sources['Natural gas'].fillna(sources['Natural Gas'], inplace=True)
sources['Large hydro'].fillna(sources['Large Hydro'], inplace=True)
sources.drop(columns=['Natural Gas', 'Large Hydro'], inplace=True)

# Linear interpolatio for missing data
sources.interpolate(method='linear', axis=0, inplace=True)

In [15]:
sources.head()

,Solar,Wind,Geothermal,Biomass,Biogas,Small hydro,Coal,Nuclear,Natural gas,Large hydro,Batteries,Imports,Other
2019-01-01 00:00:00,0.0,2810.0,993.0,380.0,225.0,200.0,11.0,2273.0,7326.0,1924.0,6.0,6254.0,0.0
2019-01-01 00:05:00,0.0,2862.0,993.0,381.0,226.0,201.0,11.0,2273.0,7200.0,1866.0,65.0,6266.0,0.0
2019-01-01 00:10:00,0.0,2916.0,993.0,380.0,226.0,202.0,11.0,2272.0,7057.0,1849.0,64.0,6319.0,0.0
2019-01-01 00:15:00,0.0,2920.0,993.0,378.0,223.0,203.0,11.0,2272.0,7007.0,1827.0,25.0,6354.0,0.0
2019-01-01 00:20:00,0.0,2902.0,993.0,379.0,223.0,203.0,11.0,2273.0,6970.0,1840.0,32.0,6360.0,0.0


In [ ]:
non_renewable = ['Coal', 'Nuclear', 'Natural gas']

df = DataFrame(sources[non_renewable], columns=non_renewable)

# create signal columns with the sin cos teqnuiche
day = 60*60*24
year = 365.2425*day
df['Seconds'] = df.index.map(Timestamp.timestamp)
df['Day sin'] = sin(df['Seconds'] * (2* pi / day))
df['Day cos'] = cos(df['Seconds'] * (2 * pi / day))
df['Year sin'] = sin(df['Seconds'] * (2 * pi / year))
df['Year cos'] = cos(df['Seconds'] * (2 * pi / year))
df = df.drop('Seconds', axis=1)

df[:10]

In [29]:
def df_to_tensor(df, window_size):
    data = df.to_numpy()
    X, y = [], []
    for i in range(data.shape[0]-window_size):
        row = [r for r in data[i:i+window_size]]
        label = [data[i+window_size][0], data[i+window_size][1], data[i+window_size][2]]
        X.append(row)
        y.append(label)
    return array(X), array(y)

In [45]:
def df_to_tensor(df, window_size, n_p, feature_cols, label_cols):
    """
    Slices an isolated dataframe into windowed X and sequence-based y.
    
    feature_cols: list of column names used as input features
    label_cols: list of column names to be predicted
    """
    # Extract underlying numpy arrays
    feature_data = df[feature_cols].to_numpy()
    label_data = df[label_cols].to_numpy()
    
    X, y = [], []
    
    # Ensure we don't out-index the array while looking ahead N_p steps
    total_steps = len(df) - window_size - n_p + 1
    
    for i in range(total_steps):
        # Input window: features from i to i + window_size
        X_window = feature_data[i : i + window_size]
        
        # Output window: labels from i + window_size to i + window_size + n_p
        y_sequence = label_data[i + window_size : i + window_size + n_p]
        
        X.append(X_window)
        y.append(y_sequence)
        
    return array(X), array(y)

In [47]:
# Define your structural parameters
WINDOW_SIZE = 8
N_P = 5 # Example: Predict 5 steps ahead
feature_cols = ['Coal', 'Nuclear', 'Natural gas', 'Day sin', 'Day cos', 'Year sin', 'Year cos']
label_cols = ['Coal', 'Nuclear', 'Natural gas'] # Adapting your original 3 labels

X_all_list = []
y_all_list = []

for i, f in enumerate(source_files):
    if stat(f).st_size == 0:
        continue
        
    # 1. Parse individual file
    temp_df = read_csv(f, header=0, nrows=288, skip_blank_lines=False)
    temp_df['Date'] = [to_date(filenames[i]) for _ in range(288)]
    temp_df.index = to_datetime(temp_df['Date'] + ' ' + temp_df['Time'], format='%Y-%m-%d %H:%M')
    temp_df.drop(columns=['Date', 'Time'], inplace=True)
    
    # 2. Handle missing columns & interpolation per file
    temp_df['Natural gas'].fillna(temp_df['Natural Gas'], inplace=True, silent=True) # avoiding warnings
    temp_df['Large hydro'].fillna(temp_df['Large Hydro'], inplace=True, silent=True)
    temp_df.drop(columns=['Natural Gas', 'Large Hydro'], errors='ignore', inplace=True)
    temp_df.interpolate(method='linear', axis=0, inplace=True)
    
    # 3. Feature engineering (sin/cos transformations)
    temp_df['Seconds'] = temp_df.index.map(Timestamp.timestamp)
    temp_df['Day sin'] = sin(temp_df['Seconds'] * (2 * pi / day))
    temp_df['Day cos'] = cos(temp_df['Seconds'] * (2 * pi / day))
    temp_df['Year sin'] = sin(temp_df['Seconds'] * (2 * pi / year))
    temp_df['Year cos'] = cos(temp_df['Seconds'] * (2 * pi / year))
    temp_df.drop('Seconds', axis=1, inplace=True)
    
    # 4. Generate tensors safely inside this dataframe boundaries
    # This guarantees windows NEVER span across file gaps
    X_file, y_file = df_to_tensor(temp_df, WINDOW_SIZE, N_P, feature_cols, label_cols)
    
    if len(X_file) > 0: # Check if file was large enough to yield windows
        X_all_list.append(X_file)
        y_all_list.append(y_file)

# 5. Concatenate the safely isolated chunks into a master dataset
X = concat(X_all_list, axis=0)
y = concat(y_all_list, axis=0)

KeyError: 'Natural Gas'

In [19]:
print(y[0])

[  11. 2274. 6789.]


In [20]:
X.shape, y.shape

((315640, 8, 7), (315640, 3))

In [37]:
# 2 full years per hour data
#X_train, y_train = X[:210527], y[:210527]
#X_test, y_test = X[210527:], y[210527:]
# X_val, y_val = X[290000:], y[290000:]
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=1)

In [38]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((211478, 8, 7), (211478, 3), (104162, 8, 7), (104162, 3))

In [39]:
# find mean and std for each dimension (except cos sin cols, they're alredy standardized)

coal_train_mean = mean(X_train[:, :, 0])
coal_train_std = std(X_train[:, :, 0])

nucl_train_mean = mean(X_train[:, :, 1])
nucl_train_std = std(X_train[:, :, 1])

ngas_train_mean = mean(X_train[:, :, 2])
ngas_train_std = std(X_train[:, :, 2])

print(coal_train_std, nucl_train_std, ngas_train_std)


def preprocess_input(X):
    X[:, :, 0] = (X[:, :, 0] - coal_train_mean) / coal_train_std
    X[:, :, 1] = (X[:, :, 1] - nucl_train_mean) / nucl_train_mean
    X[:, :, 2] = (X[:, :, 2] - ngas_train_mean) / ngas_train_mean

def preprocess_output(y):
    y[:, 0] = (y[:, 0] - coal_train_mean) / coal_train_std
    y[:, 1] = (y[:, 1] - nucl_train_mean) / nucl_train_mean
    y[:, 2] = (y[:, 2] - ngas_train_mean) / ngas_train_mean
    return y

4.576414815038189 578.5165517567001 3948.474745961745


In [40]:
preprocess_input(X_train)
preprocess_input(X_test)

print(X_train[0, 0, :])
print(X_test[0, 0, :])

[ 1.2572988   0.20454342  0.50340099  0.96004985 -0.27982901 -0.74141443
 -0.67104742]
[ 0.60176382 -0.3886273  -0.1363826  -0.88701083 -0.46174861  0.65227338
  0.7579838 ]


In [41]:
preprocess_output(y_train)
preprocess_output(y_test)

print(y_train[0, :])
print(y_test[0, :])

[1.47581046 0.20400806 0.54428443]
[ 0.60176382 -0.38809194  0.17874549]


In [42]:
model = Sequential()
model.add(InputLayer((X.shape[1], X.shape[2])))
model.add(LSTM(64))
model.add(Dense(8, 'relu', kernel_regularizer=l2(0.01), bias_regularizer=l2(0.01)))
model.add(Dense(3, 'linear', kernel_regularizer=l2(0.01), bias_regularizer=l2(0.01)))
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 64)                18432     
                                                                 
 dense (Dense)               (None, 8)                 520       
                                                                 
 dense_1 (Dense)             (None, 3)                 27        
                                                                 
Total params: 18,979
Trainable params: 18,979
Non-trainable params: 0
_________________________________________________________________


In [43]:
cp = ModelCheckpoint('model/', save_best_only=True)
es = EarlyStopping(patience=5, monitor='val_loss', mode='min', restore_best_weights=True, verbose=1)
model.compile(loss=MeanSquaredError(), optimizer=Adam(learning_rate=0.01), metrics=[RootMeanSquaredError()])

In [44]:
history = model.fit(X_train, y_train, validation_split=0.33, epochs=20, callbacks=[cp, es], shuffle=False)

Epoch 1/20
4409/4428 [============================>.] - ETA: 0s - loss: 0.0406 - root_mean_squared_error: 0.1233

INFO:tensorflow:Assets written to: model\assets


INFO:tensorflow:Assets written to: model\assets


4428/4428 [==============================] - 12s 3ms/step - loss: 0.0406 - root_mean_squared_error: 0.1233 - val_loss: 0.0419 - val_root_mean_squared_error: 0.1379
Epoch 2/20
4419/4428 [============================>.] - ETA: 0s - loss: 0.0359 - root_mean_squared_error: 0.1122

INFO:tensorflow:Assets written to: model\assets


...

INFO:tensorflow:Assets written to: model\assets


4428/4428 [==============================] - 29s 7ms/step - loss: 0.0355 - root_mean_squared_error: 0.1109 - val_loss: 0.0408 - val_root_mean_squared_error: 0.1314
Epoch 5/20
4425/4428 [============================>.] - ETA: 0s - loss: 0.0355 - root_mean_squared_error: 0.1109

INFO:tensorflow:Assets written to: model\assets


INFO:tensorflow:Assets written to: model\assets


4428/4428 [==============================] - 30s 7ms/step - loss: 0.0355 - root_mean_squared_error: 0.1109 - val_loss: 0.0400 - val_root_mean_squared_error: 0.1302
Epoch 6/20
4428/4428 [==============================] - 27s 6ms/step - loss: 0.0354 - root_mean_squared_error: 0.1104 - val_loss: 0.0405 - val_root_mean_squared_error: 0.1316
Epoch 7/20
4428/4428 [==============================] - 29s 7ms/step - loss: 0.0355 - root_mean_squared_error: 0.1109 - val_loss: 0.0404 - val_root_mean_squared_error: 0.1305
Epoch 8/20
4428/4428 [==============================] - 27s 6ms/step - loss: 0.0356 - root_mean_squared_error: 0.1111 - val_loss: 0.0409 - val_root_mean_squared_error: 0.1319
Epoch 9/20
4428/4428 [==============================] - 27s 6ms/step - loss: 0.0356 - root_mean_squared_error: 0.1112 - val_loss: 0.0401 - val_root_mean_squared_error: 0.1301
Epoch 10/20
4428/4428 [==============================] - 26s 6ms/step - loss: 0.0356 - root_mean_squared_error: 0.1113 - val_loss: 0.040